In [1]:
## Load My .env Fle For LangSmith Accessability
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import os
import pandas as pd

# QA
inputs = [
    "For customer-facing applications, which company's models dominate the top rankings?",
    "What percentage of respondents are using RAG in some form?",
    "How often are most respondents updating their models?",
]

outputs = [
    "OpenAI models dominate, with 3 of the top 5 and half of the top 10 most popular models for customer-facing apps.",
    "70% of respondents are using RAG in some form.",
    "More than 50% update their models at least monthly, with 17% doing so weekly.",
]


# Dataset
qa_pairs = [{"question": q, "answer": a} for q, a in zip(inputs, outputs)]
df = pd.DataFrame(qa_pairs)


# Move Step Back Where data folder can be found
os.chdir("../")

# Write to csv
csv_path = "data/goldens.csv"
df.to_csv(csv_path, index=False)

In [3]:
from langsmith import Client

client = Client()

dataset_name = "LLMOps-RAG-Goldens"

# Store The Created Dataset onto LangSmith Via Client
dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Input and expected output pairs for AgenticAIReport",
)

# Also Create Examples
client.create_examples(
    inputs=[{"question": q} for q in inputs],
    outputs=[{"answer": a} for a in outputs],
    dataset_id=dataset.id,
)

{'example_ids': ['afd964d8-6371-4f16-97cf-4057ef59e4fd',
  '0fd464bb-d426-4f15-baa9-1e02e9ac76c9',
  '89f65e5f-b82c-40a0-897c-a29af8a54665'],
 'count': 3}

In [ ]:
%pwd

In [5]:
from pathlib import Path
from multi_doc_chat.src.document_ingestion.data_ingestion import ChatIngestor
from multi_doc_chat.src.document_chat.retrieval import ConversationalRAG
import os


# Simple file adapter for local file paths
class LocalFileAdapter:
    """Adapter for local file paths to work with ChatIngestor."""
    def __init__(self, file_path: str):
        self.path = Path(file_path)
        self.name = self.path.name
    
    def getbuffer(self) -> bytes:
        return self.path.read_bytes()


def answer_ai_report_question(
    inputs: dict,
    data_path: str = "data/The 2025 AI Engineering Report.txt",
    chunk_size: int = 1000,
    chunk_overlap: int = 200,
    k: int = 5
) -> dict:
    """
    Answer questions about the AI Engineering Report using RAG.
    
    Args:
        inputs: Dictionary containing the question, e.g., {"question": "What is RAG?"}
        data_path: Path to the AI Engineering Report text file
        chunk_size: Size of text chunks for splitting
        chunk_overlap: Overlap between chunks
        k: Number of documents to retrieve
    
    Returns:
        Dictionary with the answer, e.g., {"answer": "RAG stands for..."}
    """
    try:
        # Extract question from inputs
        question = inputs.get("question", "")
        if not question:
            return {"answer": "No question provided"}
        
        # Check if file exists
        if not Path(data_path).exists():
            return {"answer": f"Data file not found: {data_path}"}
        
        # Create file adapter
        file_adapter = LocalFileAdapter(data_path)
        
        # Build index using ChatIngestor
        ingestor = ChatIngestor(
            temp_base="data",
            faiss_base="faiss_index",
            use_session_dirs=True
        )
        
        # Build retriever
        ingestor.built_retriver(
            uploaded_files=[file_adapter],
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            k=k
        )
        
        # Get session ID and index path
        session_id = ingestor.session_id
        index_path = f"faiss_index/{session_id}"
        
        # Create RAG instance and load retriever
        rag = ConversationalRAG(session_id=session_id)
        rag.load_retriever_from_faiss(
            index_path=index_path,
            k=k,
            index_name=os.getenv("FAISS_INDEX_NAME", "index")
        )
        
        # Get answer
        answer = rag.invoke(question, chat_history=[])
        
        return {"answer": answer}
    
    except Exception as e:
        return {"answer": f"Error: {str(e)}"}

In [6]:
# Test the function with a sample question
test_input = {"question": "For customer-facing applications, which company's models dominate the top rankings?"}

result = answer_ai_report_question(test_input)

print("Question:", test_input["question"])
print("\nAnswer:", result["answer"])

{"timestamp": "2025-10-31T11:13:24.490125Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2025-10-31T11:13:24.491865Z", "level": "info", "event": "Loaded GROQ_API_KEY from individual env var"}
{"timestamp": "2025-10-31T11:13:24.493867Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from individual env var"}
{"keys": {"GROQ_API_KEY": "gsk_dE...", "GOOGLE_API_KEY": "AIzaSy..."}, "timestamp": "2025-10-31T11:13:24.495166Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2025-10-31T11:13:24.504055Z", "level": "info", "event": "YAML config loaded"}
{"session_id": "session_20251031_131324_716f7411", "temp_dir": "data/session_20251031_131324_716f7411", "faiss_dir": "faiss_index/session_20251031_131324_716f7411", "sessionized": true, "timestamp": "2025-10-31T11:13:24.508884Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "The 2025 AI Engineering Report.txt", "saved_as

Question: For customer-facing applications, which company's models dominate the top rankings?

Answer: For customer-facing applications, OpenAI models dominate the top rankings with 3 of the top 5 and half of the top 10 most popular models.


Evaluation With LangSmith

In [7]:
# Example: Test with all golden questions

print("Testing all questions from the dataset:\n")

for i, q in enumerate(inputs, 1):
    test_input = {"question": q}
    result = answer_ai_report_question(test_input)
    print(f"Q{i}: {q}")
    print(f"A{i}: {result['answer']}\n")
    print("-" * 80 + "\n")

{"timestamp": "2025-10-31T11:13:31.142852Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2025-10-31T11:13:31.155421Z", "level": "info", "event": "Loaded GROQ_API_KEY from individual env var"}
{"timestamp": "2025-10-31T11:13:31.157817Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from individual env var"}
{"keys": {"GROQ_API_KEY": "gsk_dE...", "GOOGLE_API_KEY": "AIzaSy..."}, "timestamp": "2025-10-31T11:13:31.163487Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2025-10-31T11:13:31.183585Z", "level": "info", "event": "YAML config loaded"}
{"session_id": "session_20251031_131331_f368a59b", "temp_dir": "data/session_20251031_131331_f368a59b", "faiss_dir": "faiss_index/session_20251031_131331_f368a59b", "sessionized": true, "timestamp": "2025-10-31T11:13:31.194468Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "The 2025 AI Engineering Report.txt", "saved_as

Testing all questions from the dataset:



{"added": 1, "index": "faiss_index/session_20251031_131331_f368a59b", "timestamp": "2025-10-31T11:13:32.550140Z", "level": "info", "event": "FAISS index updated"}
{"k": 5, "fetch_k": 20, "lambda_mult": 0.5, "timestamp": "2025-10-31T11:13:32.555918Z", "level": "info", "event": "Using MMR search"}
{"timestamp": "2025-10-31T11:13:32.561797Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2025-10-31T11:13:32.563364Z", "level": "info", "event": "Loaded GROQ_API_KEY from individual env var"}
{"timestamp": "2025-10-31T11:13:32.564504Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from individual env var"}
{"keys": {"GROQ_API_KEY": "gsk_dE...", "GOOGLE_API_KEY": "AIzaSy..."}, "timestamp": "2025-10-31T11:13:32.565498Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2025-10-31T11:13:32.569588Z", "level": "info", "event": "YAML config loaded"}
{"provider": "google", "model": "gemini-2.0-fla

Q1: For customer-facing applications, which company's models dominate the top rankings?
A1: For customer-facing applications, OpenAI models dominate the top rankings with 3 of the top 5 and half of the top 10 most popular models.

--------------------------------------------------------------------------------



{"added": 1, "index": "faiss_index/session_20251031_131337_b7aeff85", "timestamp": "2025-10-31T11:13:38.566802Z", "level": "info", "event": "FAISS index updated"}
{"k": 5, "fetch_k": 20, "lambda_mult": 0.5, "timestamp": "2025-10-31T11:13:38.570119Z", "level": "info", "event": "Using MMR search"}
{"timestamp": "2025-10-31T11:13:38.575525Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2025-10-31T11:13:38.577582Z", "level": "info", "event": "Loaded GROQ_API_KEY from individual env var"}
{"timestamp": "2025-10-31T11:13:38.580745Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from individual env var"}
{"keys": {"GROQ_API_KEY": "gsk_dE...", "GOOGLE_API_KEY": "AIzaSy..."}, "timestamp": "2025-10-31T11:13:38.582420Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2025-10-31T11:13:38.590106Z", "level": "info", "event": "YAML config loaded"}
{"provider": "google", "model": "gemini-2.0-fla

Q2: What percentage of respondents are using RAG in some form?
A2: 70% of respondents are using Retrieval-Augmented Generation (RAG) in some form.

--------------------------------------------------------------------------------



{"added": 1, "index": "faiss_index/session_20251031_131341_57332336", "timestamp": "2025-10-31T11:13:44.152061Z", "level": "info", "event": "FAISS index updated"}
{"k": 5, "fetch_k": 20, "lambda_mult": 0.5, "timestamp": "2025-10-31T11:13:44.153591Z", "level": "info", "event": "Using MMR search"}
{"timestamp": "2025-10-31T11:13:44.157994Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2025-10-31T11:13:44.159043Z", "level": "info", "event": "Loaded GROQ_API_KEY from individual env var"}
{"timestamp": "2025-10-31T11:13:44.160871Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from individual env var"}
{"keys": {"GROQ_API_KEY": "gsk_dE...", "GOOGLE_API_KEY": "AIzaSy..."}, "timestamp": "2025-10-31T11:13:44.162807Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2025-10-31T11:13:44.169434Z", "level": "info", "event": "YAML config loaded"}
{"provider": "google", "model": "gemini-2.0-fla

Q3: How often are most respondents updating their models?
A3: More than 50% of practitioners update their models at least monthly, and 17% update weekly. ([Amplify Partners][2])

--------------------------------------------------------------------------------



In [ ]:
## I Do not kwon why it is not working for me, althought iam using the latest version and all is compatable ??

from langsmith.evaluation import evaluate, LangChainStringEvaluator


# Evaluators
qa_evaluator = [LangChainStringEvaluator("cot_qa")] # cot_qa -> Chain of Thought QA
dataset_name = "LLMOps-RAG-Goldens"


# Run evaluation using our RAG function
experiment_results = evaluate(
    answer_ai_report_question,
    data=dataset_name,
    evaluators=qa_evaluator,
    experiment_prefix="test-LLMOpsRAGAdvanced-qa-rag",
    # Experiment metadata
    metadata={
        "variant": "RAG with FAISS and AI Engineering Report",
        "chunk_size": 1000,
        "chunk_overlap": 200,
        "k": 5,
    },
)

### Custom Correctness Evaluator
Creating an LLM-as-a-Judge evaluator to assess semantic and factual alignment

In [9]:
from langsmith.schemas import Run, Example
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

def correctness_evaluator(run: Run, example: Example) -> dict:
    """
    Custom LLM-as-a-Judge evaluator for correctness.
    
    Correctness means how well the actual model output matches the reference output 
    in terms of factual accuracy, coverage, and meaning.
    
    Args:
        run: The Run object containing the actual outputs
        example: The Example object containing the expected outputs
    
    Returns:
        dict with 'score' (1 for correct, 0 for incorrect) and 'reasoning'
    """
    # Extract actual and expected outputs
    actual_output = run.outputs.get("answer", "")
    expected_output = example.outputs.get("answer", "")
    input_question = example.inputs.get("question", "")
    
    # Define the evaluation prompt
    eval_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are an evaluator whose job is to judge correctness.

Correctness means how well the actual model output matches the reference output in terms of factual accuracy, coverage, and meaning.

- If the actual output matches the reference output semantically (even if wording differs), it should be marked correct.
- If the output misses key facts, introduces contradictions, or is factually incorrect, it should be marked incorrect.

Do not penalize for stylistic or formatting differences unless they change meaning."""),
        ("human", """<example>
<input>
{input}
</input>

<output>
Expected Output: {expected_output}

Actual Output: {actual_output}
</output>
</example>

Please grade the following agent run given the input, expected output, and actual output.
Focus only on correctness (semantic and factual alignment).

Respond with:
1. A brief reasoning (1-2 sentences)
2. A final verdict: either "CORRECT" or "INCORRECT"

Format your response as:
Reasoning: [your reasoning]
Verdict: [CORRECT or INCORRECT]""")
    ])
    
    # Initialize LLM (using Gemini as shown in your config)
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-pro",
        temperature=0
    )
    
    # Create chain and invoke
    chain = eval_prompt | llm
    
    try:
        response = chain.invoke({
            "input": input_question,
            "expected_output": expected_output,
            "actual_output": actual_output
        })
        
        response_text = response.content
        
        # Parse the response
        reasoning = ""
        verdict = ""
        
        for line in response_text.split('\n'):
            if line.startswith("Reasoning:"):
                reasoning = line.replace("Reasoning:", "").strip()
            elif line.startswith("Verdict:"):
                verdict = line.replace("Verdict:", "").strip()
        
        # Convert verdict to score (1 for correct, 0 for incorrect)
        score = 1 if "CORRECT" in verdict.upper() else 0
        
        return {
            "key": "correctness",
            "score": score,
            "reasoning": reasoning,
            "comment": f"Verdict: {verdict}"
        }
        
    except Exception as e:
        return {
            "key": "correctness",
            "score": 0,
            "reasoning": f"Error during evaluation: {str(e)}"
        }

#### Run Evaluation with Custom Correctness Evaluator

In [11]:
# Run evaluation with the custom correctness evaluator
from langsmith.evaluation import evaluate

# Define evaluators - using custom correctness evaluator
evaluators = [correctness_evaluator]

dataset_name = "LLMOps-RAG-Goldens"

# Run evaluation
experiment_results = evaluate(
    answer_ai_report_question,
    data=dataset_name,
    evaluators=evaluators,
    experiment_prefix="LLMOpsRAGAdvanced-correctness-eval",
    description="Evaluating RAG system with custom correctness evaluator (LLM-as-a-Judge)",
    metadata={
        "variant": "RAG with FAISS and AI Engineering Report",
        "evaluator": "custom_correctness_llm_judge",
        "model": "gemini-2.5-pro",
        "chunk_size": 1000,
        "chunk_overlap": 200,
        "k": 5,
    },
)

print("\nEvaluation completed! Check the LangSmith UI for detailed results.")

{"timestamp": "2025-10-31T11:16:00.628531Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2025-10-31T11:16:00.632035Z", "level": "info", "event": "Loaded GROQ_API_KEY from individual env var"}
{"timestamp": "2025-10-31T11:16:00.634568Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from individual env var"}
{"keys": {"GROQ_API_KEY": "gsk_dE...", "GOOGLE_API_KEY": "AIzaSy..."}, "timestamp": "2025-10-31T11:16:00.636799Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2025-10-31T11:16:00.641045Z", "level": "info", "event": "YAML config loaded"}
{"session_id": "session_20251031_131600_4b25f57e", "temp_dir": "data/session_20251031_131600_4b25f57e", "faiss_dir": "faiss_index/session_20251031_131600_4b25f57e", "sessionized": true, "timestamp": "2025-10-31T11:16:00.642512Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "The 2025 AI Engineering Report.txt", "saved_as

View the evaluation results for experiment: 'LLMOpsRAGAdvanced-correctness-eval-480a4dc2' at:
https://smith.langchain.com/o/63004f96-f086-40fe-a2b6-c82532329aea/datasets/5c14c699-7dd7-4927-be6c-1a9421a000ec/compare?selectedSessions=43cb0064-17ee-4fc5-b1dd-fdeb5727a7ba




{"added": 1, "index": "faiss_index/session_20251031_131600_4b25f57e", "timestamp": "2025-10-31T11:16:02.161280Z", "level": "info", "event": "FAISS index updated"}
{"k": 5, "fetch_k": 20, "lambda_mult": 0.5, "timestamp": "2025-10-31T11:16:02.165849Z", "level": "info", "event": "Using MMR search"}
{"timestamp": "2025-10-31T11:16:02.174583Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2025-10-31T11:16:02.180319Z", "level": "info", "event": "Loaded GROQ_API_KEY from individual env var"}
{"timestamp": "2025-10-31T11:16:02.186217Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from individual env var"}
{"keys": {"GROQ_API_KEY": "gsk_dE...", "GOOGLE_API_KEY": "AIzaSy..."}, "timestamp": "2025-10-31T11:16:02.187563Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2025-10-31T11:16:02.199874Z", "level": "info", "event": "YAML config loaded"}
{"provider": "google", "model": "gemini-2.0-fla


Evaluation completed! Check the LangSmith UI for detailed results.


### Optional: Combine Multiple Evaluators
You can use multiple evaluators together to get different perspectives on your RAG system's performance.

In [ ]:
# Example: Combine custom correctness evaluator with LangChain's built-in evaluators
from langsmith.evaluation import evaluate, LangChainStringEvaluator

# Combine custom and built-in evaluators
combined_evaluators = [
    correctness_evaluator,  # Custom LLM-as-a-Judge
    LangChainStringEvaluator("cot_qa"),  # Chain-of-thought QA evaluator
]

# Run evaluation with multiple evaluators
# Uncomment to run:
# experiment_results_combined = evaluate(
#     answer_ai_report_question,
#     data=dataset_name,
#     evaluators=combined_evaluators,
#     experiment_prefix="agenticAIReport-multi-eval",
#     description="Evaluating RAG system with multiple evaluators",
#     metadata={
#         "variant": "RAG with FAISS",
#         "evaluators": "correctness + cot_qa",
#         "chunk_size": 1000,
#         "chunk_overlap": 200,
#         "k": 5,
#     },
# )